In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 1: CNN Hyperparameter Search
# Purpose: Select best hyperparameters using synthetic train/val sets only.
# The winning combination is then fixed for ALL subsequent CNN experiments
# across all datasets and conditions. Test set is never touched here.
#
# Architecture: 3 Conv + 2 FC layers [CITE SECOND ARTICLE]
# Search grid:
#   Hidden activation : {ReLU, SoftMax}
#   Optimizer         : {Adadelta, Adam, SGD}
#   Dropout rate      : {0.25, 0.5}
# Total: 2 x 3 x 2 = 12 combinations
# Epochs: 50 (fixed, as per article)
# Selection criterion: best val_accuracy

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import classification_report
import gc

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Data
IMAGE_SIZE = 28
CHANNELS   = 1    # grayscale
CLASSES    = 10

CLASS_NAMES = [
    'T-shirt/Top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot'
]

# Training — fixed as per article
BATCH_SIZE = 128
EPOCHS     = 50

# Hyperparameter search grid
# Values taken from Table 11 (Bento 2021) and [CITE SECOND ARTICLE]
ACTIVATION_OPTIONS = ['relu', 'softmax']
OPTIMIZER_OPTIONS  = ['adadelta', 'adam', 'sgd']
DROPOUT_OPTIONS    = [0.25, 0.5]

# Paths
PROJECT_DIR = '/path/to/thesis_project'
DATA_DIR    = PROJECT_DIR
SYNTH_PATH  = os.path.join(DATA_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                            'fashion_mnist', 'fashion_mnist_synthetic.npz')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'cnn', 'fashion_mnist', 'hp_search')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Data Loading ──────────────────────────────────────────────────────
# HP search uses synthetic train and val sets only.
# Real data and test set are not loaded here — they play no role in
# model selection and must not influence any hyperparameter decision.

synth_data    = np.load(SYNTH_PATH)
X_train_synth = synth_data['X_train']
y_train_synth = synth_data['y_train']
X_val_synth   = synth_data['X_val']
y_val_synth   = synth_data['y_val']

print(f'Synth train: {X_train_synth.shape}, y: {y_train_synth.shape}')
print(f'Synth val:   {X_val_synth.shape},   y: {y_val_synth.shape}')

In [ ]:
# ── Cell 3: Preprocessing ─────────────────────────────────────────────────────
# Normalize uint8 [0,255] -> float32 [0,1]

def preprocess(X):
    return X.astype('float32') / 255.0

X_train_n = preprocess(X_train_synth)
X_val_n   = preprocess(X_val_synth)

In [ ]:
# ── Cell 4: Model Builder ─────────────────────────────────────────────────────
# Architecture from [CITE SECOND ARTICLE]:
# 3x Conv2D(64 filters, 2x2) with MaxPool and Dropout
# 2x FC layers (Dense(64) + Dropout, Dense(10, softmax))
# padding='same' preserves spatial dimensions through conv layers
# hidden_activation: searched over {relu, softmax}
# dropout_rate: searched over {0.25, 0.5}
# Output activation always softmax — standard for multi-class classification

def build_model(hidden_activation, dropout_rate):
    model = models.Sequential([
        layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),

        # Block 1
        layers.Conv2D(64, kernel_size=2, padding='same',
                      activation=hidden_activation),
        layers.MaxPooling2D(pool_size=2),
        layers.Dropout(dropout_rate),

        # Block 2
        layers.Conv2D(64, kernel_size=2, padding='same',
                      activation=hidden_activation),
        layers.MaxPooling2D(pool_size=2),
        layers.Dropout(dropout_rate),

        # Block 3 — no pooling as per article
        layers.Conv2D(64, kernel_size=2, padding='same',
                      activation=hidden_activation),
        layers.Dropout(dropout_rate),

        # FC head
        layers.Flatten(),
        layers.Dense(64, activation=hidden_activation),
        layers.Dropout(dropout_rate),
        layers.Dense(CLASSES, activation='softmax'),
    ], name='cnn_3conv_2fc')
    return model


def get_optimizer(name):
    # Default parameters for all optimizers — no manual tuning
    if name == 'adadelta':
        return optimizers.Adadelta()
    elif name == 'adam':
        return optimizers.Adam()
    elif name == 'sgd':
        return optimizers.SGD()

In [ ]:
# ── Cell 5: Hyperparameter Search ─────────────────────────────────────────────
# Train each of the 12 combinations on synthetic train set.
# Evaluate on synthetic val set after each epoch.
# Record best val_accuracy per run.
# No test set involved at any point.

search_results = []

for activation in ACTIVATION_OPTIONS:
    for opt_name in OPTIMIZER_OPTIONS:
        for dropout in DROPOUT_OPTIONS:

            run_name = f'act{activation}_opt{opt_name}_do{dropout}'
            print(f"\n{'='*60}")
            print(f'Run: {run_name}')
            print(f"{'='*60}")

            # Clear session — no leakage across runs
            tf.keras.backend.clear_session()
            gc.collect()

            model = build_model(activation, dropout)
            model.compile(
                optimizer=get_optimizer(opt_name),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )

            history = model.fit(
                X_train_n, y_train_synth,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                validation_data=(X_val_n, y_val_synth),
                verbose=1
            )

            best_val_acc   = max(history.history['val_accuracy'])
            final_val_acc  = history.history['val_accuracy'][-1]
            final_train_acc= history.history['accuracy'][-1]
            

            print(f'  Final train acc: {final_train_acc:.4f}')
            print(f'  Final val acc:   {final_val_acc:.4f}')

            # Save run history
            run_dir = os.path.join(RESULTS_DIR, run_name)
            os.makedirs(run_dir, exist_ok=True)
            pd.DataFrame(history.history).to_csv(
                os.path.join(run_dir, 'history.csv'), index_label='epoch'
            )

            # Plot training curves
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(history.history['loss'],     label='train loss')
            axes[0].plot(history.history['val_loss'], label='val loss')
            axes[0].set_title('Loss')
            axes[0].set_xlabel('Epoch')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(history.history['accuracy'],     label='train accuracy')
            axes[1].plot(history.history['val_accuracy'], label='val accuracy')
            axes[1].set_title('Accuracy')
            axes[1].set_xlabel('Epoch')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)

            plt.suptitle(f'Training Curves — {run_name}', fontsize=11)
            plt.tight_layout()
            plt.savefig(os.path.join(run_dir, 'training_curves.png'),
                        dpi=150, bbox_inches='tight')
            plt.close()

            search_results.append({
                'run_name':        run_name,
                'activation':      activation,
                'optimizer':       opt_name,
                'dropout':         dropout,
                'best_val_acc':    round(best_val_acc,    4),
                'final_train_acc': round(final_train_acc, 4),
                'final_val_acc':   round(final_val_acc,   4),
            })

            # Save after each run — safe against crashes
            pd.DataFrame(search_results).to_csv(
                os.path.join(RESULTS_DIR, 'search_results.csv'), index=False
            )

In [ ]:
# ── Cell 6: Summary & Best Hyperparameters ────────────────────────────────────

results_df = pd.DataFrame(search_results)

print(f"\n{'='*70}")
print('Hyperparameter Search Results — Ranked by Best Val Accuracy')
print(f"{'='*70}")
print(results_df.sort_values('best_val_acc', ascending=False)
               .to_string(index=False))

best_run = results_df.loc[results_df['best_val_acc'].idxmax()]

print(f"\n{'='*70}")
print('Best Hyperparameters')
print(f"{'='*70}")
print(f'  Activation : {best_run["activation"]}')
print(f'  Optimizer  : {best_run["optimizer"]}')
print(f'  Dropout    : {best_run["dropout"]}')
print(f'  Best val acc: {best_run["best_val_acc"]}')
print(f"\nThese hyperparameters are fixed for all subsequent CNN experiments.")
print(f"Record them before proceeding to notebooks 2 and 3.")

# Save summary
results_df.sort_values('best_val_acc', ascending=False).to_csv(
    os.path.join(RESULTS_DIR, 'search_results_ranked.csv'), index=False
)
print(f'\nSaved: {RESULTS_DIR}/search_results_ranked.csv')

In [ ]:
# ── Cell 7: Summary Plots ─────────────────────────────────────────────────────

results_df = pd.read_csv(os.path.join(RESULTS_DIR, 'search_results_ranked.csv'))
best_run_name = results_df.iloc[0]['run_name']

# Plot 1 — Val accuracy all 12 runs
fig, ax = plt.subplots(figsize=(14, 6))
for _, row in results_df.iterrows():
    hist = pd.read_csv(os.path.join(RESULTS_DIR, row['run_name'], 'history.csv'))
    lw        = 2.5 if row['run_name'] == best_run_name else 1.0
    alpha     = 1.0 if row['run_name'] == best_run_name else 0.5
    ax.plot(hist['val_accuracy'], label=row['run_name'], linewidth=lw, alpha=alpha)

ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy')
ax.set_title('Val Accuracy — All 12 Search Runs (DDIM FashionMNIST)')
ax.legend(fontsize=7, loc='lower right', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'val_accuracy_all_runs.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: val_accuracy_all_runs.png')

# Plot 2 — Train and val accuracy for best run
best_hist = pd.read_csv(os.path.join(RESULTS_DIR, best_run_name, 'history.csv'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(best_hist['loss'],     label='train loss')
axes[0].plot(best_hist['val_loss'], label='val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(best_hist['accuracy'],     label='train accuracy')
axes[1].plot(best_hist['val_accuracy'], label='val accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Training Curves — Best Run: {best_run_name}', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'best_run_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: best_run_curves.png')